In [0]:
import json
import time
import pandas as pd
import requests

dbutils.widgets.text("gemini_api_key", "COLAR AQUI", "Chave da API Gemini")  # (name, defaultValue, label)
API_KEY = dbutils.widgets.get("gemini_api_key").strip()

# Carrega a amostra original e verifica se já existe progresso salvo (checkpoint)
df_reviews = spark.table("silver.reviews_amostra_ia").toPandas()
tabela_destino = "silver.reviews_enriquecidas_ia"

if spark.catalog.tableExists(tabela_destino):
    ids_prontos = set(spark.table(tabela_destino).select("review_id").toPandas()["review_id"])
    df_pendentes = df_reviews[~df_reviews["review_id"].isin(ids_prontos)].copy()
    print(f"Retomando: {len(ids_prontos)} já salvas | {len(df_pendentes)} pendentes.")
else:
    df_pendentes = df_reviews.copy()
    print(f"Iniciando: {len(df_pendentes)} avaliações para classificar.")

# Modelos priorizados conforme painel de Limite de Taxa Google AI Studio, por custo X benefício
modelos_disponiveis = [
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-3.8-flash"
]

def chamar_gemini_com_seguranca(prompt_texto):
    payload = {
        "contents": [{"parts": [{"text": prompt_texto}]}],
        "generationConfig": {"responseMimeType": "application/json"}
    }
    for modelo in modelos_disponiveis:
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{modelo}:generateContent?key={API_KEY}"
        for tentativa in range(1, 4):  # Até 3 tentativas antes de trocar de modelo
            try:
                resp = requests.post(url, json=payload, timeout=90)
                if resp.status_code == 200:
                    texto = resp.json()["candidates"][0]["content"]["parts"][0]["text"]
                    return json.loads(texto), modelo
                elif resp.status_code in (429, 503):
                    espera = 15 * tentativa
                    print(f"Aviso HTTP {resp.status_code} no {modelo}. Aguardando {espera}s...")
                    time.sleep(espera)
                else:
                    print(f"Modelo {modelo} indisponível (HTTP {resp.status_code}). Alternando...")
                    break
            except Exception as erro:
                print(f"Falha de rede ({erro}). Tentando novamente em 10s...")
                time.sleep(10)
    raise RuntimeError("Todos os modelos falharam após retentativas. O progresso anterior já está salvo.")

# Processa em 6 lotes de 50 avaliações (= 6 RPD)
tamanho_lote = 50
total_pendentes = len(df_pendentes)

for inicio in range(0, total_pendentes, tamanho_lote):
    lote_completo = df_pendentes.iloc[inicio : inicio + tamanho_lote].copy()
    lista_entrada = lote_completo[["review_id", "comentario"]].to_dict(orient="records")

    prompt = f"""
    Classifique as avaliações de e-commerce abaixo.
    Retorne uma lista JSON onde CADA objeto tenha exatamente estas 4 chaves:
    - "review_id": mantenha o mesmo ID recebido
    - "sentimento": apenas "Positivo", "Neutro" ou "Negativo"
    - "categoria_problema": apenas "Atraso na Entrega", "Produto Danificado", "Divergencia de Anuncio", "Atendimento" ou "Sem Problema"
    - "resumo_curto": resumo objetivo em no máximo 6 palavras

    Avaliações:
    {json.dumps(lista_entrada, ensure_ascii=False)}
    """

    dados_lote, modelo_usado = chamar_gemini_com_seguranca(prompt)

    # Cruza e salva o lote imediatamente na tabela Delta (garantindo zero desperdício)
    df_classificado = pd.DataFrame(dados_lote)
    df_lote_final = pd.merge(lote_completo, df_classificado, on="review_id", how="inner")

    for col in ["sentimento", "categoria_problema", "resumo_curto"]:
        df_lote_final[col] = df_lote_final[col].fillna("Não informado").astype(str)

    spark.createDataFrame(df_lote_final).write.mode("append").saveAsTable(tabela_destino)

    concluidos = min(inicio + tamanho_lote, total_pendentes)
    print(f"[{modelo_usado}] Lote salvo: {concluidos}/{total_pendentes} avaliações.")

    # Pausa de 6s mantém ~10 RPM (abaixo do teto de 15 RPM)
    if concluidos < total_pendentes:
        time.sleep(6)

print("Concluído! Todas as avaliações foram classificadas e salvas em 'silver.reviews_enriquecidas_ia'.")

In [0]:
%sql
SELECT sentimento, categoria_problema, COUNT(*) AS qtd
FROM silver.reviews_enriquecidas_ia
GROUP BY 1, 2
ORDER BY qtd DESC;